# Dataset Split: Random

Random 80/10/10 split evaluation for the compact hydrogen-bond statistical model.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-mprl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("outputs/hbond_analysis")
PLOTS_DIR = BASE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = BASE_DIR / "hbond_length_disentanglement_table.csv"
RANDOM_STATE = 7

SELECTED_FEATURES = [
    "sequence_length",
    "hbond_per_residue",
    "seq_class_nonlocal_per_residue",
    "strong_nonlocal_fraction",
    "strong_nonlocal_per_residue",
    "nonlocal_backbone_backbone_per_residue",
    "hbond_contact_order",
]

print(DATA_PATH.resolve())

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["PDB_ID", "Sequence", "v127", "v128"] + SELECTED_FEATURES).copy().reset_index(drop=True)
df["log1p_v127"] = np.log1p(df["v127"].astype(float))
df["log1p_v128"] = np.log1p(df["v128"].astype(float))

train_idx, temp_idx = train_test_split(df.index.to_numpy(), test_size=0.2, random_state=RANDOM_STATE)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=RANDOM_STATE)
split = {"train": sorted(map(int, train_idx)), "val": sorted(map(int, val_idx)), "test": sorted(map(int, test_idx))}

for name, indices in split.items():
    pd.Series(df.loc[indices, "PDB_ID"].to_numpy()).to_csv(BASE_DIR / f"dataset_random_{name}_pdb_ids.txt", index=False, header=False)

print({name: len(indices) for name, indices in split.items()})

In [ ]:
def build_models():
    return {
        "length_only_ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
        "selected_hbond_ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
        "selected_hbond_random_forest": Pipeline([
            ("scaler", StandardScaler()),
            ("model", MultiOutputRegressor(RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=3,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ))),
        ]),
        "selected_hbond_extra_trees": Pipeline([
            ("scaler", StandardScaler()),
            ("model", MultiOutputRegressor(ExtraTreesRegressor(
                n_estimators=500,
                min_samples_leaf=3,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ))),
        ]),
    }


feature_sets = {
    "length_only_ridge": ["sequence_length"],
    "selected_hbond_ridge": SELECTED_FEATURES,
    "selected_hbond_random_forest": SELECTED_FEATURES,
    "selected_hbond_extra_trees": SELECTED_FEATURES,
}


def evaluate_predictions(true_raw, pred_raw):
    row = {}
    for j, target_name in enumerate(["toughness_v127", "strength_v128"]):
        row[f"{target_name}/r2"] = float(r2_score(true_raw[:, j], pred_raw[:, j]))
        row[f"{target_name}/mae"] = float(mean_absolute_error(true_raw[:, j], pred_raw[:, j]))
        row[f"{target_name}/rmse"] = float(np.sqrt(mean_squared_error(true_raw[:, j], pred_raw[:, j])))
        row[f"{target_name}/spearman"] = float(spearmanr(true_raw[:, j], pred_raw[:, j]).correlation)
    row["mean/r2"] = float(np.mean([row["toughness_v127/r2"], row["strength_v128/r2"]]))
    row["mean/spearman"] = float(np.mean([row["toughness_v127/spearman"], row["strength_v128/spearman"]]))
    return row


X = df[SELECTED_FEATURES].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
y_log = df[["log1p_v127", "log1p_v128"]].to_numpy(dtype=float)
y_raw = df[["v127", "v128"]].to_numpy(dtype=float)

rows = []
pred_frames = []
models = build_models()
for model_name, model in models.items():
    cols = feature_sets[model_name]
    model.fit(X.loc[split["train"], cols], y_log[split["train"]])
    for split_name, indices in split.items():
        pred_raw = np.expm1(model.predict(X.loc[indices, cols]))
        true_raw = y_raw[indices]
        row = {"dataset_split": "random", "model": model_name, "split": split_name, "n": int(len(indices)), "n_features": len(cols)}
        row.update(evaluate_predictions(true_raw, pred_raw))
        rows.append(row)
        pred_frames.append(pd.DataFrame({
            "dataset_split": "random",
            "model": model_name,
            "split": split_name,
            "PDB_ID": df.loc[indices, "PDB_ID"].to_numpy(),
            "true_v127": true_raw[:, 0],
            "true_v128": true_raw[:, 1],
            "pred_v127": pred_raw[:, 0],
            "pred_v128": pred_raw[:, 1],
        }))

metrics_df = pd.DataFrame(rows)
predictions_df = pd.concat(pred_frames, ignore_index=True)
metrics_df.to_csv(BASE_DIR / "dataset_random_hbond_model_metrics.csv", index=False)
predictions_df.to_csv(BASE_DIR / "dataset_random_hbond_model_predictions.csv", index=False)
display(metrics_df[metrics_df["split"] == "test"].sort_values("mean/r2", ascending=False))

In [ ]:
summary = {
    "split_method": "random",
    "seed": RANDOM_STATE,
    "split_sizes": {name: len(indices) for name, indices in split.items()},
    "selected_features": SELECTED_FEATURES,
    "metrics_path": str(BASE_DIR / "dataset_random_hbond_model_metrics.csv"),
    "predictions_path": str(BASE_DIR / "dataset_random_hbond_model_predictions.csv"),
}
(BASE_DIR / "dataset_random_hbond_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary